In [2]:
from functools import lru_cache
import time

# ============================================================
# Problem data
# ============================================================

beta = 1 / 1.05

P = {
    0: {0: 0.10, 1: 0.45, 2: 0.45},
    1: {0: 0.20, 1: 0.20, 2: 0.60},
    2: {0: 0.10, 1: 0.70, 2: 0.20},
}

DELTA = {
    0: 0,
    1: 10,
    2: -10,
}

SHORT_FEE_RATE = 0.01

# state = (z, price, cash, stocks, futures, shorts_due, forced_no_trade, bankrupt)
INITIAL_STATE = (0, 100, 100, 1, 0, 0, 0, 0)

# ============================================================
# Speed/solver settings
# ============================================================

# True  -> much faster, heuristic only
# False -> broader action set, more exact
HEURISTIC_EXTREME_ACTIONS = True

# If True, compare only states with same time + market structure
USE_DOMINANCE = True

# Optional coarse cash bucketing for even more speed in very large T
# Set to 1 for exact cash, or e.g. 5 / 10 for approximate bucketing.
CASH_BUCKET = 1

# ============================================================
# Utilities
# ============================================================

def bucket_cash(cash):
    if CASH_BUCKET <= 1:
        return int(cash)
    return int(cash // CASH_BUCKET) * CASH_BUCKET


def state_to_str(state):
    z, price, cash, stocks, futures, shorts_due, forced_no_trade, bankrupt = state
    return {
        "z": z,
        "price": price,
        "cash": cash,
        "stocks": stocks,
        "futures": futures,
        "shorts_due": shorts_due,
        "forced_no_trade": forced_no_trade,
        "bankrupt": bankrupt,
    }


def normalize_state(state):
    """
    Canonicalise state for memoization.
    """
    z, price, cash, stocks, futures, shorts_due, forced_no_trade, bankrupt = state

    cash = bucket_cash(cash)

    # prevent weird negative inventory bugs from spreading
    stocks = max(0, int(stocks))
    futures = max(0, int(futures))
    shorts_due = max(0, int(shorts_due))
    forced_no_trade = int(bool(forced_no_trade))
    bankrupt = int(bool(bankrupt))

    return (int(z), int(price), int(cash), stocks, futures, shorts_due, forced_no_trade, bankrupt)


def stage_reward(state_after_action):
    """
    Reward at end of day t = cash at hand.
    """
    return state_after_action[2]


def net_wealth_mark_to_market(state):
    """
    A rough monotone proxy for state quality.
    Futures are valued at current spot for dominance comparisons.
    Shorts due are liabilities at current spot.
    """
    z, price, cash, stocks, futures, shorts_due, forced_no_trade, bankrupt = state
    return cash + (stocks + futures - shorts_due) * price


def worst_case_next_cash(state):
    """
    Monotone lower bound on next-day cash after mandatory settlements.
    Useful for dominance pruning.

    Worst case:
    - longs / futures suffer price drop
    - shorts suffer price rise
    """
    z, price, cash, stocks, futures, shorts_due, forced_no_trade, bankrupt = state

    if bankrupt:
        return -10**18

    next_price_worst = price

    # for shorts due, worst next price is +10
    if shorts_due > 0:
        next_price_worst = price + 10

    wcash = cash

    # futures settle tomorrow at today's agreed price
    if futures > 0:
        wcash -= futures * price

    # shorts must be bought back tomorrow at worst-case higher spot
    if shorts_due > 0:
        wcash -= shorts_due * (price + 10)

    return wcash


def apply_next_day_price(state, next_change):
    """
    Transition from end of day t to start/end-of-day-(before decision) t+1:
      - price moves
      - futures mature
      - shorts_due are forcibly settled
    """
    z, price, cash, stocks, futures, shorts_due, forced_no_trade, bankrupt = state

    if bankrupt:
        return normalize_state((next_change, price, cash, stocks, futures, shorts_due, 0, 1))

    new_price = price + DELTA[next_change]
    forced_no_trade = 0

    # futures mature: pay agreed price = today's current price
    if futures > 0:
        cash -= futures * price
        stocks += futures
        futures = 0
        if cash < 0:
            return normalize_state((next_change, new_price, cash, stocks, futures, shorts_due, 0, 1))

    # shorts_due must be settled tomorrow, no other trading that day
    if shorts_due > 0:
        cash -= shorts_due * new_price
        shorts_due = 0
        forced_no_trade = 1
        if cash < 0:
            return normalize_state((next_change, new_price, cash, stocks, futures, shorts_due, forced_no_trade, 1))

    return normalize_state((next_change, new_price, cash, stocks, futures, shorts_due, forced_no_trade, 0))


# ============================================================
# Actions
# ============================================================

def feasible_short_max(cash, price):
    """
    Shorting rule from question:
    need enough money to buy back next day at current price * 1.01
    per your interpretation / discussion, use 1.01 * current price as bound.
    """
    if price <= 0:
        return 0

    required_per_share = 1.01 * price
    return int(cash // required_per_share)


def enumerate_actions(state):
    """
    Return feasible actions.

    Heuristic mode:
      keep only extreme actions:
        none
        sell all stock
        buy max futures
        short max

    Broader mode:
      enumerate all feasible k values.
    """
    z, price, cash, stocks, futures, shorts_due, forced_no_trade, bankrupt = state

    if bankrupt or forced_no_trade or shorts_due > 0:
        return [("none", 0)]

    actions = [("none", 0)]

    if price <= 0:
        return actions

    # Sell stock
    if stocks > 0:
        if HEURISTIC_EXTREME_ACTIONS:
            actions.append(("sell_stock", stocks))
        else:
            for k in range(1, stocks + 1):
                actions.append(("sell_stock", k))

    # Buy futures
    max_buy_future = int(cash // price)
    if max_buy_future > 0:
        if HEURISTIC_EXTREME_ACTIONS:
            actions.append(("buy_future", max_buy_future))
        else:
            for k in range(1, max_buy_future + 1):
                actions.append(("buy_future", k))

    # Shorting allowed only when currently no stocks and no futures
    if stocks == 0 and futures == 0:
        max_short = feasible_short_max(cash, price)
        if max_short > 0:
            if HEURISTIC_EXTREME_ACTIONS:
                actions.append(("short", max_short))
            else:
                for k in range(1, max_short + 1):
                    actions.append(("short", k))

    return actions


def apply_action(state, action):
    """
    Apply action immediately at end of current day.
    """
    z, price, cash, stocks, futures, shorts_due, forced_no_trade, bankrupt = state
    a_type, k = action

    if bankrupt or forced_no_trade or shorts_due > 0:
        return state

    if a_type == "none":
        return state

    if a_type == "sell_stock":
        if k > stocks:
            return normalize_state((z, price, cash, stocks, futures, shorts_due, forced_no_trade, 1))
        cash += k * price
        stocks -= k
        return normalize_state((z, price, cash, stocks, futures, shorts_due, forced_no_trade, 0))

    if a_type == "buy_future":
        if k * price > cash:
            return normalize_state((z, price, cash, stocks, futures, shorts_due, forced_no_trade, 1))
        futures += k
        return normalize_state((z, price, cash, stocks, futures, shorts_due, forced_no_trade, 0))

    if a_type == "short":
        if stocks != 0 or futures != 0:
            return normalize_state((z, price, cash, stocks, futures, shorts_due, forced_no_trade, 1))

        fee = k * price * SHORT_FEE_RATE
        proceeds = k * price

        if fee > cash:
            return normalize_state((z, price, cash, stocks, futures, shorts_due, forced_no_trade, 1))

        cash = cash - fee + proceeds
        shorts_due += k
        return normalize_state((z, price, cash, stocks, futures, shorts_due, forced_no_trade, 0))

    raise ValueError(f"Unknown action {action}")


# ============================================================
# Dominance pruning
# ============================================================

dominance_frontier = {}

def dominance_key(state, t, T):
    """
    Only compare states that have same timing / market / forced constraints.
    """
    z, price, cash, stocks, futures, shorts_due, forced_no_trade, bankrupt = state
    return (t, T, z, price, futures, shorts_due, forced_no_trade, bankrupt)


def dominance_features(state):
    """
    State A dominates B if it has:
    - weakly more cash
    - weakly more stock
    - weakly better marked wealth
    - weakly better worst-case next cash
    with at least one strict
    """
    z, price, cash, stocks, futures, shorts_due, forced_no_trade, bankrupt = state
    return (
        cash,
        stocks,
        net_wealth_mark_to_market(state),
        worst_case_next_cash(state),
    )


def dominates(fA, fB):
    return (
        fA[0] >= fB[0] and
        fA[1] >= fB[1] and
        fA[2] >= fB[2] and
        fA[3] >= fB[3] and
        fA != fB
    )


def is_dominated(state, t, T):
    global dominance_frontier

    if not USE_DOMINANCE:
        return False

    key = dominance_key(state, t, T)
    feat = dominance_features(state)
    frontier = dominance_frontier.get(key, [])

    for old_feat in frontier:
        if dominates(old_feat, feat):
            return True

    new_frontier = []
    for old_feat in frontier:
        if not dominates(feat, old_feat):
            new_frontier.append(old_feat)

    new_frontier.append(feat)
    dominance_frontier[key] = new_frontier
    return False


# ============================================================
# DP
# ============================================================

memo = {}
policy_cache = {}

def action_priority(action):
    """
    Try 'more liquidating / stronger move' actions first.
    Helps pruning.
    """
    a, k = action
    order = {
        "sell_stock": 0,
        "short": 1,
        "none": 2,
        "buy_future": 3,
    }
    return (order[a], -k)


def dp_value(state, t, T):
    global memo, cache_hits

    state = normalize_state(state)
    key = (state, t, T)

    if key in memo:
        return memo[key]

    if t > T:
        return 0.0

    if state[-1] == 1:  # bankrupt
        memo[key] = 0.0
        return memo[key]

    if is_dominated(state, t, T):
        memo[key] = 0.0
        return memo[key]

    best_val = float("-inf")
    best_action = None

    actions = sorted(enumerate_actions(state), key=action_priority)

    for action in actions:
        s_after = apply_action(state, action)
        immediate = (beta ** t) * stage_reward(s_after)

        if t == T:
            total = immediate
        else:
            continuation = 0.0
            curr_z = s_after[0]

            for z_next, prob in P[curr_z].items():
                s_next = apply_next_day_price(s_after, z_next)
                continuation += prob * dp_value(s_next, t + 1, T)

            total = immediate + continuation

        if total > best_val:
            best_val = total
            best_action = action

    memo[key] = best_val
    policy_cache[key] = best_action
    return best_val


def dp_policy(state, t, T):
    state = normalize_state(state)
    key = (state, t, T)

    if key not in policy_cache:
        _ = dp_value(state, t, T)

    return policy_cache.get(key), memo.get(key)


def print_policy_tree_dp(state, t, T, indent=0, max_depth=4):
    prefix = " " * indent
    action, val = dp_policy(state, t, T)
    print(f"{prefix}t={t}, state={state_to_str(state)}, best_action={action}, value={val:.4f}")

    if t >= T or max_depth <= 1 or action is None:
        return

    s_after = apply_action(state, action)
    curr_z = s_after[0]

    for z_next, prob in P[curr_z].items():
        s_next = apply_next_day_price(s_after, z_next)
        print(f"{prefix}  -> next z={z_next} with prob={prob}")
        print_policy_tree_dp(s_next, t + 1, T, indent + 6, max_depth - 1)


# ============================================================
# Exhaustive search (small T only)
# ============================================================

def exhaustive_value(state, t, T):
    """
    Warning: only usable for small T.
    """
    state = normalize_state(state)

    if t > T:
        return 0.0, None

    best_val = float("-inf")
    best_action = None

    for action in enumerate_actions(state):
        s_after = apply_action(state, action)
        immediate = (beta ** t) * stage_reward(s_after)

        if t == T:
            total = immediate
        else:
            continuation = 0.0
            curr_z = s_after[0]
            for z_next, prob in P[curr_z].items():
                s_next = apply_next_day_price(s_after, z_next)
                v_next, _ = exhaustive_value(s_next, t + 1, T)
                continuation += prob * v_next
            total = immediate + continuation

        if total > best_val:
            best_val = total
            best_action = action

    return best_val, best_action


# ============================================================
# Runner helpers
# ============================================================

def reset_solver():
    global memo, policy_cache, dominance_frontier
    memo = {}
    policy_cache = {}
    dominance_frontier = {}

def run_dp(state, T):
    reset_solver()
    start = time.time()
    val = dp_value(state, 0, T)
    elapsed = time.time() - start

    return {
        "value": val,
        "time_sec": elapsed,
    }


def run_exhaustive(state, T):
    start = time.time()
    val, action = exhaustive_value(state, 0, T)
    elapsed = time.time() - start
    return {
        "value": val,
        "best_action_t0": action,
        "time_sec": elapsed,
    }

In [ ]:
T_large = 100
print("\n" + "=" * 70)
print("DP for T = 100")
print("=" * 70)
res_large = run_dp(INITIAL_STATE, T_large)
print(res_large)


DP for T = 100
{'value': 897.8544433646473, 'time_sec': 0.002361297607421875}
